# Neljännesvuosittainen pro forma -tuloslaskelma PROC COMPUTAB -proseduurilla


## Tiivistelmä

Tässä muistikirjassa rakennetaan alueellisen pankin neljännesvuosittainen pro forma -tuloslaskelma suoraan kuukausittaisesta pääkirja-aineistosta käyttäen PROC COMPUTAB -proseduuria, SAS/ETS:n raportinkirjoituksen taulukkoproseduuria. Se reitittää kunkin kuukauden korkotuotot, korkokulut, palkkiotuotot ja käyttökulut oikeaan kalenterineljänneksen sarakkeeseen, laskee sitten riviohjelmointilohkoilla nettokorkotuoton, tuloksen ennen veroja ja nettotuloksen, sekä sarakelohkolla kokoaa neljä vuosineljännestä koko vuoden summaksi.

## Tietolähteet

Yksittäinen synteettinen aineisto `bank_ledger` simuloi keskikokoisen paikallispankin yhden tilikauden kuukausittaisia tilinpäätöserien rivejä. Kaksitoista kuukausittaista havaintoa generoidaan koodissa komennoilla `call streaminit`/`rand`, joten muistikirja on täysin itsenäinen.

| Muuttuja | Tyyppi | Kuvaus |
|----------|------|-------------|
| `stmtdate` | Num (DATE9.) | Kuukauden lopun tilinpäätöspäivä (tammi–joulu) |
| `loanint`  | Num | Lainaportfoliosta ansaitut korot ja palkkiot (tuhatta USD) |
| `secint`   | Num | Sijoitusarvopaperikannasta ansaittu korko (tuhatta USD) |
| `depint`   | Num | Talletuksille maksettu korko (tuhatta USD) |
| `borrint`  | Num | Lainanotoista / FHLB-ennakoista maksettu korko (tuhatta USD) |
| `feeinc`   | Num | Korottomat (palkkio- ja palvelumaksu-) tuotot (tuhatta USD) |
| `salaries` | Num | Palkka- ja henkilöstöetuuskulut (tuhatta USD) |
| `occupancy`| Num | Toimitila- ja laitekulut (tuhatta USD) |
| `othropex` | Num | Muut korottomat käyttökulut (tuhatta USD) |
| `provision`| Num | Luottotappiovaraus (tuhatta USD) |
| `taxrate`  | Num | Tulokseen ennen veroja sovellettu efektiivinen veroaste |

# Neljännesvuosittainen pro forma -tuloslaskelma PROC COMPUTAB -proseduurilla

Pankkien talousryhmät muuntavat rutiininomaisesti kuukausittaisen pääkirjan **neljännesvuosittaiseksi tuloslaskelmaksi**, joka näyttää nettokorkotuoton ja viivan alle jäävän nettotuloksen. `PROC COMPUTAB` (SAS/ETS) on tähän tarkoitukseen rakennettu: se asettelee ohjelmoitavan taulukon, jonka **sarakkeet** ovat raportointijaksot ja **rivit** tilinpäätöserien rivit, ja antaa laskea välisummat tavanomaisella DATA-askeleen logiikalla rivi- ja sarakelohkoissa.

Tässä muistikirjassa me:

1. Generoimme yhden tilikauden synteettistä kuukausittaista pääkirja-aineistoa paikallispankille.
2. Reititämme kunkin kuukauden sen kalenterineljännekseen ja rakennamme neljännesvuosittaisen tuloslaskelman.
3. Laskemme nettokorkotuoton, tuloksen ennen veroja ja nettotuloksen **rivilohkossa**, ja kokoamme vuosineljännekset koko vuoden summaksi **sarakelohkossa**.
4. Käytämme uudelleen `out=`-taulukkoa, jotta laskettu laskelma voi syöttää jatkoraportointia.

## Vaihe 1 — Generoi synteettinen kuukausittainen pääkirja-aineisto

Simuloimme kaksitoista kuukauden lopun havaintoa. Laina- ja arvopaperikorkotuotot nousevat loivasti vuoden aikana, talletus- ja lainanottokulut skaalautuvat korkoympäristön mukaan, ja palkkiotuotot, käyttökulut ja luottotappiovaraus kantavat realistista kuukaudesta toiseen vaihtelevaa kohinaa. `call streaminit` kiinnittää siemenluvun, joten data on toistettavissa.

In [1]:
TIEDOT bank_ledger;
   CALL streaminit(20240115);
   MUOTO stmtdate date9.;
   TEE month = 1 ASTI 12;
      /* Kuukauden lopun tilinpäätöspäivä tilikaudelle 2024 */
      stmtdate = mdy(month, 28, 2024);

      /* Loiva nouseva trendi vuoden aikana + kuukausikohina */
      drift = 1 + 0.012 * (month - 1);

      /* Korkotuotot (tuhatta USD) */
      loanint = round(1850 * drift + rand('normal', 0, 45), 0.01);
      secint  = round( 420 * drift + rand('normal', 0, 15), 0.01);

      /* Korkokulut (tuhatta USD) */
      depint  = round( 540 * drift + rand('normal', 0, 22), 0.01);
      borrint = round( 130 * drift + rand('normal', 0, 10), 0.01);

      /* Korottomat tuotot ja kulut (tuhatta USD) */
      feeinc    = round(310 + rand('normal', 0, 18), 0.01);
      salaries  = round(720 + rand('normal', 0, 25), 0.01);
      occupancy = round(165 + rand('normal', 0, 8),  0.01);
      othropex  = round(240 + rand('normal', 0, 20), 0.01);

      /* Luottotappiovaraus, ajoittain koholla */
      provision = round(95 + rand('exponential') * 40, 0.01);

      /* Efektiivinen veroaste */
      taxrate = 0.21;

      TULOSTE;
   LOPPU;
   SÄILYTÄ stmtdate loanint secint depint borrint
        feeinc salaries occupancy othropex provision taxrate;
SUORITA;

PROSEDUURI TULOSTA TIEDOT=bank_ledger noobs NIMIKE;
   OTSIKKO 'Synteettinen kuukausittainen pääkirja — paikallispankki tilikausi 2024';
   NIMIKE stmtdate  = 'Tilinpäätöspäivä'
          loanint   = 'Lainojen korot'
          secint    = 'Arvopaperikorot'
          depint    = 'Talletuskorot'
          borrint   = 'Lainanottokorot'
          feeinc    = 'Palkkiotuotot'
          salaries  = 'Palkat'
          occupancy = 'Toimitilat'
          othropex  = 'Muut kulut'
          provision = 'Luottotappiovaraus'
          taxrate   = 'Veroaste';
   MUOTO loanint secint depint borrint feeinc
          salaries occupancy othropex provision comma8.2;
SUORITA;

                         Synteettinen kuukausittainen pääkirja — paikallispankki tilikausi 2024                         

     Tilinpäätöspäivä  Lainojen korot  Arvopaperikorot  Talletuskorot  Lainanottokorot  Palkkiotuotot  Palkat  Toimitilat  Muut kulut  Luottotappiovaraus  Veroaste
            28JAN2024        1,772.95           423.79         531.47           128.99         339.33  699.38      171.95      202.43              130.93      0.21
            28FEB2024        1,824.38           421.13         564.85           125.53         294.09  767.29      162.79      307.61              123.25      0.21
            28MAR2024        1,931.98           442.06         536.64           131.71         295.72  724.03      153.26      254.21              115.76      0.21
            28APR2024        1,934.99           439.29         535.94           140.01         294.56  729.47      174.19      237.43              198.05      0.21
            28MAY2024        1,949.31           447.35    


NOTE: DATA bank_ledger


NOTE: Wrote bank_ledger (12 rows, 11 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=bank_ledger

NOTE: PROC PRINT completed: 12 observations printed, 11 variables


## Vaihe 2 — Rakenna neljännesvuosittainen tuloslaskelma

Raportin ydin on alla oleva `PROC COMPUTAB` -askel.

- **`columns qtr1 qtr2 qtr3 qtr4 fy;`** määrittää neljä vuosineljännessaraketta sekä koko vuoden sarakkeen.
- Nimeämätön **syöttölohko** asettaa automaattisen muuttujan **`_col_`** arvolla `qtr(stmtdate)`, joka reitittää kunkin kuukausittaisen havainnon oikeaan vuosineljännessarakkeeseen. Koska syöte transponoidaan oletuksena, kukin pääkirjamuuttuja päätyy riville, jolla on sama nimi.
- **Rivilohko** `inc_stmt:` ajetaan kerran saraketta kohti ja laskee johdetut rivit — nettokorkotuoton, korottomien kulujen kokonaismäärän, tuloksen ennen veroja ja nettotuloksen — täsmälleen kuten kirjanpitäjä tekisi.
- **Sarakelohko** `total:` ajetaan kerran riviä kohti ja summaa neljä vuosineljännestä `fy`-sarakkeeseen (koko vuosi).

`rows ... / ...` -lauseet lisäävät otsikot, sisennykset ja viivarivit (`ol` yliviiva, `ul` alaviiva, `dul` kaksoisalaviiva), jotta tuloste lukeutuu kuin todellinen tilinpäätös.

In [2]:
OTSIKKO 'Pro forma -tuloslaskelma — Community Bancorp, Inc.';
title2 'Tilikausi 2024  (Määrät tuhansina USD)';

PROSEDUURI computab TIEDOT=bank_ledger cspace=2 cwidth=11 out=qtr_income;

   /* Neljä vuosineljännestä sekä koottu koko vuoden sarake */
   columns qtr1 qtr2 qtr3 qtr4 fy / MUOTO=comma11.1;
   columns qtr1 / '1. nelj.';
   columns qtr2 / '2. nelj.';
   columns qtr3 / '3. nelj.';
   columns qtr4 / '4. nelj.';
   columns fy   / 'Koko' 'vuosi' +3;

   /* Tuloslaskelman rivit, ylhäältä alas */
   rows loanint  / 'Lainojen korot ja palkkiot';
   rows secint   / 'Arvopaperikorot'               ul;
   rows intinc   / 'Korkotuotot yhteensä';
   rows depint   / 'Talletuskorot';
   rows borrint  / 'Lainanottokorot'               ul;
   rows intexp   / 'Korkokulut yhteensä';
   rows nii      / 'Nettokorkotuotto'              ol skip;
   rows provision/ 'Luottotappiovaraus'            ul;
   rows niiap    / 'Nettokorkot. varauksen jälkeen' skip;
   rows feeinc   / 'Korottomat tuotot'             ul;
   rows salaries / 'Palkat ja etuudet';
   rows occupancy/ 'Toimitilat ja laitteet';
   rows othropex / 'Muut käyttökulut'              ul;
   rows nonintexp/ 'Korottomat kulut yhteensä'     skip;
   rows pretax   / 'Tulos ennen veroja'            ol;
   rows taxexp   / 'Tuloverokulu'                  ul;
   rows netinc   / 'Nettotulos'                    dul;

   /* Syöttölohko: reititä kukin kuukausi kalenterineljännekseensä */
   _col_ = qtr(stmtdate);

   /* Rivilohko: laske laskelman välisummat jokaisessa sarakkeessa */
   inc_stmt:
      intinc    = loanint + secint;
      intexp    = depint + borrint;
      nii       = intinc - intexp;
      niiap     = nii - provision;
      nonintexp = salaries + occupancy + othropex;
      pretax    = niiap + feeinc - nonintexp;
      taxexp    = pretax * 0.21;
      netinc    = pretax - taxexp;

   /* Sarakelohko: kokoa neljä vuosineljännestä koko vuoden sarakkeeseen */
   total:
      fy = qtr1 + qtr2 + qtr3 + qtr4;
SUORITA;

                                   Pro forma -tuloslaskelma — Community Bancorp, Inc.                                   
                                         Tilikausi 2024  (Määrät tuhansina USD)                                         


                             The COMPUTAB Procedure                             

                       1. nelj.     2. nelj.     3. nelj.     4. nelj.         Koko  
                                                                              vuosi  
                           qtr1         qtr2         qtr3         qtr4           fy  
                    -----------  -----------  -----------  -----------  -----------  
  Lainojen korot ja palkkiot
  loanint               5,529.3      5,818.7      5,963.5      6,277.0     23,588.4  
  Arvopaperikorot
  secint                1,287.0      1,334.9      1,342.0      1,452.9      5,416.8  
                    -----------  -----------  -----------  -----------  -----------  
  Korkotuotot yhteensä
  


NOTE: Option TITLE changed to Pro forma -tuloslaskelma — Community Bancorp, Inc..
NOTE: Option TITLE2 changed to Tilikausi 2024  (Määrät tuhansina USD).
NOTE: PROC COMPUTAB
NOTE: COMPUTAB OUT= dataset written to: qtr_income.csv
NOTE: PROC COMPUTAB statement used.


## Vaihe 3 — Käytä uudelleen COMPUTAB-tulostusaineistoa

Yllä oleva `PROC COMPUTAB` -askel kirjoitti lasketun taulukkonsa aineistoon `qtr_income` `out=`-option kautta. Kukin tuon aineiston rivi on tilinpäätöserän rivi ja kukin sarakemuuttuja (`qtr1`–`qtr4`, `fy`) sisältää lasketun arvon, joten se voi syöttää jatkoraportointia. Alla tulostamme kootun koko vuoden sarakkeen vahvistaaksemme, että luvut täsmäävät.

In [3]:
OTSIKKO 'COMPUTAB-tulosaineisto — koko vuoden sarake';

PROSEDUURI TULOSTA TIEDOT=qtr_income NIMIKE noobs;
   MUUTTUJA _row_ fy;
   MUOTO fy comma12.1;
   NIMIKE _row_ = 'Erä' fy = 'Koko vuosi';
SUORITA;

OTSIKKO;

                                      COMPUTAB-tulosaineisto — koko vuoden sarake                                       
                                         Tilikausi 2024  (Määrät tuhansina USD)                                         


      Erä  Koko vuosi
---------  ----------
loanint      23,588.4
secint        5,416.8
intinc       29,005.2
depint        6,831.2
borrint       1,650.7
intexp        8,482.0
nii          20,523.2
provision     1,568.9
niiap        18,954.3
feeinc        3,703.2
salaries      8,763.1
occupancy     1,985.6
othropex      2,944.8
nonintexp    13,693.5
pretax        8,964.1
taxexp        1,882.5
netinc        7,081.6




NOTE: Option TITLE changed to COMPUTAB-tulosaineisto — koko vuoden sarake.
NOTE: PROC PRINT data=qtr_income

NOTE: PROC PRINT completed: 17 observations printed, 2 variables


## Tulosten tulkinta

Pro forma -laskelma lukeutuu ylhäältä alas kuin pankin viranomaisraportti (call report): korkotuottojen kokonaismäärästä vähennetään korkokulujen kokonaismäärä, jolloin saadaan **nettokorkotuotto** — tässä noin 20,5 miljoonaa dollaria vuodessa — pankin ensisijainen tulonlähde. Vähentämällä **luottotappiovarauksen**, lisäämällä **palkkiotuotot** ja nettouttamalla **korottomat kulut** saadaan tulos ennen veroja, joka on noin 9,0 miljoonaa dollaria, ja soveltamalla 21 %:n efektiivistä veroastetta tuotetaan **nettotulos** lähellä 7,1 miljoonaa dollaria. `total:`-sarakelohko lisää neljä vuosineljännestä koko vuoden sarakkeeseen, joten vuosisummat täsmäävät vuosineljännesten summaan rakenteellisesti — vahvistettu `out=`-aineistossa, jossa koko vuoden `netinc` 7 081,6 vastaa neljän neljännesvuosiluvun summaa.

Koska kaikki lasketaan `PROC COMPUTAB` -proseduurin ohjelmoitavissa rivi- ja sarakelohkoissa, todellisen kuukausittaisen pääkirjan vaihtaminen sisään ei vaadi mitään muutosta raporttilogiikkaan — vain syöttöaineisto vaihtuu. `out=`-aineisto tekee sitten lasketun laskelman saataville koontinäyttöihin, trendianalyysiin tai hallituspaketin automaatioon.